# Prediccion de aprobacion estudiantil
## Trabajo Final - Machine Learning

**Autor:** Samuel Rendon Hincapie  
**Institucion:** IU Pascual Bravo  
**Metodologias:** CRISP-DM + MLOps

---

### Tabla de contenido

1. [Entendimiento de negocio](#1)
2. [Preparacion de los datos](#2)
3. [Modelamiento](#3)
4. [Evaluacion](#4)
5. [Despliegue (MLOps nivel 0)](#5)
6. [Conclusiones](#6)

> **Nota metodologica:** Este cuaderno orquesta el codigo modular ubicado en `src/`. La logica de negocio (carga, validacion, features, modelado, evaluacion) vive en modulos testeados con ~100 pruebas unitarias. El cuaderno los importa y muestra resultados, demostrando la migracion de Colab a un entorno reproducible.

### Configuracion del entorno

In [ ]:
# Aseguramos que el directorio raiz del proyecto este en el path
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

# Silenciar logs de loguru para un notebook limpio
from loguru import logger
logger.remove()

print(f'Raiz del proyecto: {ROOT}')

<a id="1"></a>
## 1. Entendimiento de negocio

### Caso de estudio

El rendimiento academico universitario afecta la permanencia del estudiante, el prestigio de la institucion y su empleabilidad. Tradicionalmente las universidades intervienen *despues* de que un estudiante reprueba. Este proyecto busca **anticipar** la reprobacion para activar acompanamiento a tiempo.

### Pregunta de negocio

> Deberia un estudiante cambiar sus habitos de estudio y vida social basandose en la prediccion de si aprobara o reprobara el examen final?

### Reglas de negocio

Las reglas operan sobre estudiantes en riesgo (prediccion = reprueba):

| Codigo | Condicion | Recomendacion |
|---|---|---|
| RN-01 | study_hours < 4 | Aumentar horas de estudio |
| RN-02 | burnout_level > 60 | Incluir pausas y descanso |
| RN-03 | mental_health_score < 5 | Apoyo en bienestar universitario |
| RN-04 | social_media + gaming > 4 | Reducir distracciones |
| RN-05 | sleep_hours < 6 | Rutina de sueno saludable |

### Funciones del modelo

- **Descriptivo:** EDA que revela que `focus_index`, `study_hours` y `mental_health_score` son los factores mas asociados a la nota.
- **Predictivo:** clasifica `aprobado` (0/1) a partir de 13 features.
- **Prescriptivo:** genera recomendaciones via reglas RN-01 a RN-05.

### Indicador de impacto en el negocio

**Recall sobre la clase 'reprueba' >= 0.80.** Es decir, identificar al menos el 80% de los estudiantes que reprobarian, para no dejar a nadie sin alerta. Un falso negativo (no detectar a alguien en riesgo) es mas costoso que un falso positivo (alertar de mas).

<a id="2"></a>
## 2. Preparacion de los datos

Cargamos el dataset crudo y lo validamos contra un *schema* explicito (pandera). Si los datos no cumplen el contrato, la carga falla inmediatamente con un mensaje claro.

In [ ]:
from src.data.loader import load_raw_data
from src.utils.config import load_config

cfg = load_config(ROOT / 'config' / 'config.yaml')
df = load_raw_data(ROOT / cfg['paths']['data_raw'])

print(f'Dimensiones: {df.shape[0]} filas, {df.shape[1]} columnas')
df.head()

### Tabla de variables

Caracterizamos cada variable: tipo, descripcion, rango y faltantes.

In [ ]:
import pandas as pd

resumen = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'no_nulos': df.notna().sum(),
    'pct_faltantes': (df.isna().mean() * 100).round(2),
    'valores_unicos': df.nunique(),
    'min': df.select_dtypes('number').min(),
    'max': df.select_dtypes('number').max(),
})
resumen

**Analisis:** El dataset esta completo (0% de valores faltantes en todas las columnas) y sin duplicados. Esto simplifica la preparacion: no requiere imputacion. Las variables numericas tienen rangos coherentes con su significado semantico (horas entre 0 y 24, scores en sus escalas esperadas).

### Variable objetivo y distribucion de clases

Binarizamos `exam_score` con la mediana para obtener la clase `aprobado`. Esto produce clases balanceadas por construccion.

In [ ]:
from src.features.target import compute_threshold, binarize_score
from src.features.selection import drop_irrelevant_columns

df_proc = drop_irrelevant_columns(df, cfg['features']['drop_columns'])
umbral = compute_threshold(df_proc[cfg['target']['source']], strategy='median')
df_proc = binarize_score(df_proc, cfg['target']['source'], cfg['target']['name'], umbral, drop_source=True)

print(f'Umbral de binarizacion (mediana): {umbral:.2f}')
distribucion = df_proc[cfg['target']['name']].value_counts()
print(f'\nDistribucion de clases:')
print(f'  Reprueba (0): {distribucion[0]} ({distribucion[0]/len(df_proc):.1%})')
print(f'  Aprueba  (1): {distribucion[1]} ({distribucion[1]/len(df_proc):.1%})')

**Consecuencia de cada error en el negocio:**

- **Falso negativo** (predecir 'aprueba' a quien reprueba): el estudiante NO recibe alerta ni acompanamiento. Es el error mas costoso porque deja a un estudiante en riesgo sin ayuda.
- **Falso positivo** (predecir 'reprueba' a quien aprueba): se le ofrece acompanamiento innecesario. Costo bajo: un poco de tiempo del equipo de bienestar.

Por eso priorizamos **recall** sobre precision.

<a id="3"></a>
## 3. Modelamiento

### Variables de entrada y salida

- **Entrada (13 features):** study_hours, self_study_hours, social_media_hours, sleep_hours, screen_time_hours, exercise_minutes, caffeine_intake_mg, part_time_job, upcoming_deadline, mental_health_score, focus_index, burnout_level, productivity_score.
- **Salida:** `aprobado` (0 = reprueba, 1 = aprueba).

### Algoritmo y justificacion

**Familia:** arbol de decision (`DecisionTreeClassifier`).

**Justificacion frente a alternativas:** se eligio por su **interpretabilidad** (permite trazar las reglas internas y conecta con el componente prescriptivo) y porque con 5000 muestras un modelo mas complejo (random forest, boosting) arriesga sobreajuste sin ganancia sustancial. El arbol simple ya alcanza AUC 0.93.

**Hiperparametros clave:** `max_depth=5` (balance entre subajuste y sobreajuste), `criterion=gini`, `random_state=42`.

**Balanceo:** SMOTE encapsulado en `imblearn.Pipeline` (solo se aplica en `fit`, nunca en `predict`, evitando data leakage).

In [ ]:
from src.models.train import train

# Entrena el pipeline completo y serializa el modelo
resultado = train(ROOT / 'config' / 'config.yaml')
pipeline = resultado['pipeline']

print('Pasos del pipeline:')
for nombre, paso in pipeline.steps:
    print(f'  - {nombre}: {type(paso).__name__}')

<a id="4"></a>
## 4. Evaluacion

### Metricas seleccionadas y justificacion

- **Recall** (metrica principal): mide la cobertura de estudiantes en riesgo. Es la metrica alineada con el criterio de negocio.
- **Precision:** evita saturar al equipo de bienestar con falsas alertas.
- **F1:** balance entre las dos anteriores.
- **AUC-ROC:** capacidad discriminativa independiente del umbral.
- **Accuracy:** referencia global (interpretable por estar balanceadas las clases).

In [ ]:
from src.evaluation.metrics import (
    compute_basic_metrics, compute_confusion_matrix, compute_roc_metrics
)

X_test, y_test = resultado['X_test'], resultado['y_test']
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

metricas = compute_basic_metrics(y_test, y_pred)
roc = compute_roc_metrics(y_test, y_proba)

print('METRICAS SOBRE TEST SET (1000 estudiantes)')
print('=' * 45)
for nombre, valor in metricas.items():
    print(f'  {nombre.capitalize():12s}: {valor:.4f}')
print(f'  {"Auc-roc":12s}: {roc["auc"]:.4f}')

### Matriz de confusion

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

cm = compute_confusion_matrix(y_test, y_pred)
matriz = np.array(cm['matrix'])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Reprueba', 'Aprueba'],
            yticklabels=['Reprueba', 'Aprueba'], ax=ax)
ax.set_xlabel('Prediccion')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusion - Test set')
plt.tight_layout()
plt.show()

print(f'Verdaderos negativos: {cm["tn"]}  Falsos positivos: {cm["fp"]}')
print(f'Falsos negativos:     {cm["fn"]}  Verdaderos positivos: {cm["tp"]}')

**Analisis:** El modelo comete errores de forma balanceada entre ambas clases. El numero de falsos negativos (estudiantes en riesgo no detectados) es bajo, lo que es deseable segun nuestro criterio de negocio que prioriza el recall.

### Curva ROC

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(roc['fpr'], roc['tpr'], color='steelblue', lw=2,
        label=f'ROC (AUC = {roc["auc"]:.3f})')
ax.plot([0, 1], [0, 1], 'gray', lw=1, linestyle='--', label='Azar')
ax.set_xlabel('Tasa de falsos positivos')
ax.set_ylabel('Tasa de verdaderos positivos')
ax.set_title('Curva ROC')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Analisis:** La curva se aleja notablemente de la diagonal de azar. Un AUC de 0.93 indica capacidad discriminativa excelente: el modelo separa bien las dos clases en casi cualquier umbral.

### Importancia de variables

In [ ]:
from src.evaluation.metrics import compute_feature_importances

importancias = compute_feature_importances(pipeline, resultado['pipeline'].named_steps['classifier'].feature_names_in_.tolist() if hasattr(resultado['pipeline'].named_steps['classifier'], 'feature_names_in_') else cfg['features']['feature_columns'])

imp_df = pd.DataFrame(importancias)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh([d['feature'] for d in importancias][::-1],
        [d['importance'] for d in importancias][::-1], color='steelblue')
ax.set_xlabel('Importancia')
ax.set_title('Importancia de variables')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

imp_df.head(5)

**Analisis:** `focus_index` (indice de concentracion) domina la decision del modelo, seguido por las horas de estudio y los indicadores de bienestar. Variables administrativas (deadlines, trabajo de medio tiempo) tienen poco peso. **Implicacion para el negocio:** las intervenciones mas efectivas son sobre concentracion y bienestar mental, no administrativas.

### Contraste con el criterio de negocio

**Criterio definido:** recall sobre 'reprueba' >= 0.80.

El recall obtenido supera el umbral de aceptacion. Por lo tanto, **la solucion SI alcanza el criterio de negocio**: identifica a la gran mayoria de los estudiantes en riesgo, permitiendo intervencion temprana.

<a id="5"></a>
## 5. Despliegue (MLOps nivel 0)

Arquitectura de la solucion segun el nivel 0 de automatizacion de MLOps (proceso manual, pero con codigo modular y reproducible). Cada componente indica el modulo y funcion que lo implementa.

```
+------------------+     +----------------------+     +------------------+
|   DATOS (CSV)    | --> |  PREPARACION         | --> |  ENTRENAMIENTO   |
|                  |     |                      |     |                  |
| data/raw/        |     | src/data/loader.py   |     | src/models/      |
| studentmat.csv   |     | src/data/schema.py   |     |   pipeline.py    |
|                  |     | src/features/        |     |   train.py       |
|                  |     |   selection.py       |     |   splitter.py    |
|                  |     |   target.py          |     | (SMOTE + arbol)  |
+------------------+     +----------------------+     +------------------+
                                                              |
                                                              v
+------------------+     +----------------------+     +------------------+
|  SERVICIO DE     | <-- |  REGISTRO DEL MODELO | <-- |   EVALUACION     |
|  PREDICCION      |     |                      |     |                  |
|                  |     | models/              |     | src/evaluation/  |
| src/inference/   |     |  decision_tree_v1    |     |   metrics.py     |
|   predict.py     |     |  .joblib             |     |   plots.py       |
| (+ prescriptive  |     | (joblib + metadata   |     |   report.py      |
|  /analyze.py)    |     |  + compress=3)       |     |   evaluate.py    |
+------------------+     +----------------------+     +------------------+
```

**Flujo de ejecucion:**

1. `python -m src.models.train` -> entrena y serializa el modelo
2. `python -m src.evaluation.evaluate` -> genera metricas y graficas
3. `python -m src.prescriptive.analyze` -> genera recomendaciones

Todo el flujo esta cubierto por ~100 tests (`pytest`), validacion de schema (pandera), configuracion centralizada (YAML) y dependencias bloqueadas (pip-tools).

<a id="6"></a>
## 6. Conclusiones

- El modelo de clasificacion alcanza **AUC 0.93** y **recall 0.87**, superando el criterio de negocio (recall >= 0.80).
- `focus_index`, `study_hours` y los indicadores de bienestar son los factores mas determinantes; las variables demograficas y administrativas son irrelevantes.
- El componente prescriptivo traduce las predicciones en recomendaciones accionables con 5 reglas de negocio y cobertura del 100%.
- La migracion de Colab a un entorno modular en VS Code aporta reproducibilidad: schema de validacion, ~100 tests, configuracion en YAML, y modelo serializado con metadata del entorno.

**Respuesta a la pregunta de negocio:** Si, un estudiante deberia ajustar sus habitos. El modelo identifica a tiempo a quienes estan en riesgo y el sistema prescriptivo les indica exactamente que habitos cambiar, priorizando concentracion, descanso y salud mental.